<p style="text-align:center">
    <a href="https://skills.network/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMDS0321ENSkillsNetwork26802033-2022-01-01" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Hands-on Lab: Interactive Visual Analytics with Folium**


Estimated time needed: **40** minutes


The launch success rate may depend on many factors such as payload mass, orbit type, and so on. It may also depend on the location and proximities of a launch site, i.e., the initial position of rocket trajectories. Finding an optimal location for building a launch site certainly involves many factors and hopefully we could discover some of the factors by analyzing the existing launch site locations.<br>
La tasa de éxito del lanzamiento puede depender de muchos factores, como la masa de la carga útil, el tipo de órbita, etc. También puede depender de la ubicación y las proximidades del lugar de lanzamiento, es decir, la posición inicial de las trayectorias de los cohetes. Encontrar una ubicación óptima para construir un sitio de lanzamiento ciertamente implica muchos factores y, con suerte, podríamos descubrir algunos de los factores analizando las ubicaciones de los sitios de lanzamiento existentes.


In the previous exploratory data analysis labs, you have visualized the SpaceX launch dataset using `matplotlib` and `seaborn` and discovered some preliminary correlations between the launch site and success rates. In this lab, you will be performing more interactive visual analytics using `Folium`.<br>
En los laboratorios de análisis de datos exploratorios anteriores, visualizó el conjunto de datos del lanzamiento de SpaceX utilizando "matplotlib" y "seaborn" y descubrió algunas correlaciones preliminares entre el sitio de lanzamiento y las tasas de éxito. En esta práctica de laboratorio, realizará análisis visuales más interactivos utilizando "Folium".


## Objectives


This lab contains the following tasks:

*   **TASK 1:** Mark all launch sites on a map
*   **TASK 2:** Mark the success/failed launches for each site on the map
*   **TASK 3:** Calculate the distances between a launch site to its proximities

After completed the above tasks, you should be able to find some geographical patterns about launch sites.


Este laboratorio contiene las siguientes tareas:

* **TAREA 1:** Marcar todos los sitios de lanzamiento en un mapa
* **TAREA 2:** Marque los lanzamientos exitosos/fallidos de cada sitio en el mapa.
* **TAREA 3:** Calcular las distancias entre un sitio de lanzamiento y sus proximidades.

Después de completar las tareas anteriores, debería poder encontrar algunos patrones geográficos sobre los sitios de lanzamiento.


Let's first import required Python packages for this lab:


In [1]:
import piplite
await piplite.install(['folium'])
await piplite.install(['pandas'])

In [2]:
import folium
import pandas as pd

In [3]:
# Import folium MarkerCluster plugin
from folium.plugins import MarkerCluster
# Import folium MousePosition plugin
from folium.plugins import MousePosition
# Import folium DivIcon plugin
from folium.features import DivIcon

If you need to refresh your memory about folium, you may download and refer to this previous folium lab:

Si necesita refrescar su memoria sobre folium, puede descargar y consultar este laboratorio de folium anterior:

[Generating Maps with Python](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/labs/v4/DV0101EN-Exercise-Generating-Maps-in-Python.ipynb)


In [ ]:
## Task 1: Mark all launch sites on a map
## Tarea 1: marcar todos los sitios de lanzamiento en un mapa


First, let's try to add each site's location on a map using site's latitude and longitude coordinates<br>
Primero, intentemos agregar la ubicación de cada sitio en un mapa usando las coordenadas de latitud y longitud del sitio.

The following dataset with the name `spacex_launch_geo.csv` is an augmented dataset with latitude and longitude added for each site.


In [4]:
# Download and read the `spacex_launch_geo.csv`
from js import fetch
import io

URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv'
resp = await fetch(URL)
spacex_csv_file = io.BytesIO((await resp.arrayBuffer()).to_py())
spacex_df=pd.read_csv(spacex_csv_file)

Now, you can take a look at what are the coordinates for each site.
<br>
Ahora, puedes echar un vistazo a cuáles son las coordenadas de cada sitio.

In [5]:
# Select relevant sub-columns: `Launch Site`, `Lat(Latitude)`, `Long(Longitude)`, `class`
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


Above coordinates are just plain numbers that can not give you any intuitive insights about where are those launch sites. If you are very good at geography, you can interpret those numbers directly in your mind. If not, that's fine too. Let's visualize those locations by pinning them on a map.<br>
Las coordenadas anteriores son simplemente números que no pueden brindarle ninguna idea intuitiva sobre dónde están esos sitios de lanzamiento. Si eres muy bueno en geografía, podrás interpretar esos números directamente en tu mente. Si no, también está bien. Visualicemos esas ubicaciones fijándolas en un mapa.


We first need to create a folium `Map` object, with an initial center location to be NASA Johnson Space Center at Houston, Texas.<br>
Primero necesitamos crear un objeto "Mapa" de folio, con una ubicación central inicial que será el Centro Espacial Johnson de la NASA en Houston, Texas.


In [6]:
# Start location is NASA Johnson Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=10)

We could use `folium.Circle` to add a highlighted circle area with a text label on a specific coordinate. For example,<br>
Podríamos usar `folium.Circle` para agregar un área circular resaltada con una etiqueta de texto en una coordenada específica. Por ejemplo,


In [7]:
# Create a blue circle at NASA Johnson Space Center's coordinate with a popup label showing its name
circle = folium.Circle(nasa_coordinate, radius=1000, color='#d35400', fill=True).add_child(folium.Popup('NASA Johnson Space Center'))
# Create a blue circle at NASA Johnson Space Center's coordinate with a icon showing its name
marker = folium.map.Marker(
    nasa_coordinate,
    # Create an icon as a text label
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'NASA JSC',
        )
    )
site_map.add_child(circle)
site_map.add_child(marker)

and you should find a small yellow circle near the city of Houston and you can zoom-in to see a larger circle.<br>
y deberías encontrar un pequeño círculo amarillo cerca de la ciudad de Houston y puedes acercarte para ver un círculo más grande.


Now, let's add a circle for each launch site in data frame `launch_sites`<br>
Ahora, agreguemos un círculo para cada sitio de lanzamiento en el marco de datos `launch_sites`.


*TODO:*  Create and add `folium.Circle` and `folium.Marker` for each launch site on the site map<br>
*TODO:* Cree y agregue `folium.Circle` y `folium.Marker` para cada sitio de lanzamiento en el mapa del sitio


An example of folium.Circle:


`folium.Circle(coordinate, radius=1000, color='#000000', fill=True).add_child(folium.Popup(...))`


An example of folium.Marker:


`folium.map.Marker(coordinate, icon=DivIcon(icon_size=(20,20),icon_anchor=(0,0), html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'label', ))`


In [12]:
# Initial the map
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)
# For each launch site, add a Circle object based on its coordinate (Lat, Long) values. In addition, add Launch site name as a popup label
# Para cada sitio de lanzamiento, agregue un objeto Círculo según sus valores de coordenadas (Lat, Long).
# Además, agregue el nombre del sitio de lanzamiento como una etiqueta emergente

for index, row in launch_sites_df.iterrows():
    # extraer los datos de la fila actual
    nombre = row['Launch Site']
    coordenada = [row['Lat'], row['Long']]
    
    # Crear y anadir el circulño con su popup
    circle = folium.Circle(coordenada, radius=1000,color='#000000',fill=True)
    circle.add_child(folium.Popup(nombre))
    site_map.add_child(circle)

    # Crear y anadir el marcador de texto (DivIcon)
    marker = folium.map.Marker(
            coordenada,
            icon=DivIcon(
                icon_size=(20,20),
                icon_anchor=(0,0),
                html=f'<div style="font-size:12px; color:#d35400;"><b>{nombre}</b></div>' 
                )
            )
    site_map.add_child(marker)

site_map

The generated map with marked launch sites should look similar to the following:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_markers.png">
</center>


Now, you can explore the map by zoom-in/out the marked areas
, and try to answer the following questions:

*   Are all launch sites in proximity to the Equator line?
*   Are all launch sites in very close proximity to the coast?

Also please try to explain your findings.

Ahora puedes explorar el mapa acercando o alejando las áreas marcadas.
, e intente responder las siguientes preguntas:

* ¿Están todos los sitios de lanzamiento cerca de la línea ecuatorial?
* ¿Están todos los sitios de lanzamiento muy cerca de la costa?

También intente explicar sus hallazgos.


In [15]:
# Task 2: Mark the success/failed launches for each site on the map
# Tarea 2: marcar los lanzamientos exitosos/fallidos de cada sitio en el mapa


Next, let's try to enhance the map by adding the launch outcomes for each site, and see which sites have high success rates.
Recall that data frame spacex_df has detailed launch records, and the `class` column indicates if this launch was successful or not <br>
A continuación, intentemos mejorar el mapa agregando los resultados del lanzamiento de cada sitio y veamos qué sitios tienen altas tasas de éxito.
Recuerde que el marco de datos spacex_df tiene registros de lanzamiento detallados y la columna "clase" indica si este lanzamiento fue exitoso o no.


In [14]:
spacex_df.tail(10)

,Launch Site,Lat,Long,class
46,KSC LC-39A,28.573255,-80.646895,1
47,KSC LC-39A,28.573255,-80.646895,1
48,KSC LC-39A,28.573255,-80.646895,1
49,CCAFS SLC-40,28.563197,-80.576820,1
50,CCAFS SLC-40,28.563197,-80.576820,1
51,CCAFS SLC-40,28.563197,-80.576820,0
52,CCAFS SLC-40,28.563197,-80.576820,0
53,CCAFS SLC-40,28.563197,-80.576820,0
54,CCAFS SLC-40,28.563197,-80.576820,1
55,CCAFS SLC-40,28.563197,-80.576820,0


Next, let's create markers for all launch records.
If a launch was successful `(class=1)`, then we use a green marker and if a launch was failed, we use a red marker `(class=0)`
<br>
A continuación, creemos marcadores para todos los registros de lanzamiento.
Si un lanzamiento fue exitoso `(class=1)`, entonces usamos un marcador verde y si un lanzamiento falló, usamos un marcador rojo `(class=0)`

Note that a launch only happens in one of the four launch sites, which means many launch records will have the exact same coordinate. Marker clusters can be a good way to simplify a map containing many markers having the same coordinate.<br>
Tenga en cuenta que un lanzamiento solo ocurre en uno de los cuatro sitios de lanzamiento, lo que significa que muchos registros de lanzamiento tendrán exactamente las mismas coordenadas. Los grupos de marcadores pueden ser una buena manera de simplificar un mapa que contiene muchos marcadores que tienen la misma coordenada.


Let's first create a `MarkerCluster` object


In [16]:
marker_cluster = MarkerCluster()


*TODO:* Create a new column in `spacex_df` dataframe called `marker_color` to store the marker colors based on the `class` value<br>
*TODO:* Cree una nueva columna en el marco de datos `spacex_df` llamada `marker_color` para almacenar los colores del marcador según el valor de `class`


In [17]:

# Apply a function to check the value of `class` column
# If class=1, marker_color value will be green
# If class=0, marker_color value will be red
def assign_marker_color(launch_outcome):
    if launch_outcome == 1:
        return 'green'
    else:
        return 'red'

spacex_df['marker_color'] = spacex_df['class'].apply(assign_marker_color)
spacex_df.tail(10)


,Launch Site,Lat,Long,class,marker_color
46,KSC LC-39A,28.573255,-80.646895,1,green
47,KSC LC-39A,28.573255,-80.646895,1,green
48,KSC LC-39A,28.573255,-80.646895,1,green
49,CCAFS SLC-40,28.563197,-80.576820,1,green
50,CCAFS SLC-40,28.563197,-80.576820,1,green
51,CCAFS SLC-40,28.563197,-80.576820,0,red
52,CCAFS SLC-40,28.563197,-80.576820,0,red
53,CCAFS SLC-40,28.563197,-80.576820,0,red
54,CCAFS SLC-40,28.563197,-80.576820,1,green
55,CCAFS SLC-40,28.563197,-80.576820,0,red


*TODO:* For each launch result in `spacex_df` data frame, add a `folium.Marker` to `marker_cluster`
<br>
*TODO:* Para cada resultado de lanzamiento en el marco de datos `spacex_df`, agregue un `folium.Marker` a `marker_cluster`

In [18]:
# Add marker_cluster to current site_map
site_map.add_child(marker_cluster)

# for each row in spacex_df data frame
# create a Marker object with its coordinate
# and customize the Marker's icon property to indicate if this launch was successed or failed, 
# e.g., icon=folium.Icon(color='white', icon_color=row['marker_color']
for index, record in spacex_df.iterrows():
    # TODO: Create and add a Marker cluster to the site map
    # marker = folium.Marker(...)
    marker = folium.Marker([record['Lat'],record['Long']],
            icon=folium.Icon(color='white', icon_color=record['marker_color'])
            )
    marker_cluster.add_child(marker)

site_map

Your updated map may look like the following screenshots:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster.png">
</center>


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster_zoomed.png">
</center>


From the color-labeled markers in marker clusters, you should be able to easily identify which launch sites have relatively high success rates.<br>
A partir de los marcadores etiquetados con colores en los grupos de marcadores, debería poder identificar fácilmente qué sitios de lanzamiento tienen tasas de éxito relativamente altas.


In [19]:
# TASK 3: Calculate the distances between a launch site to its proximities
# TAREA 3: Calcular las distancias entre un sitio de lanzamiento y sus proximidades

Next, we need to explore and analyze the proximities of launch sites.
<br>
A continuación, debemos explorar y analizar las proximidades de los sitios de lanzamiento.

Let's first add a `MousePosition` on the map to get coordinate for a mouse over a point on the map. As such, while you are exploring the map, you can easily find the coordinates of any points of interests (such as railway)<br>
Primero agreguemos una `MousePosition` en el mapa para obtener las coordenadas para pasar el mouse sobre un punto en el mapa. Como tal, mientras explora el mapa, puede encontrar fácilmente las coordenadas de cualquier punto de interés (como el ferrocarril).


In [20]:
# Add Mouse Position to get the coordinate (Lat, Long) for a mouse over on the map
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)

site_map.add_child(mouse_position)
site_map

Now zoom in to a launch site and explore its proximity to see if you can easily find any railway, highway, coastline, etc. Move your mouse to these points and mark down their coordinates (shown on the top-left) in order to the distance to the launch site.<br>
Ahora acérquese a un sitio de lanzamiento y explore su proximidad para ver si puede encontrar fácilmente algún ferrocarril, carretera, línea costera, etc. Mueva el mouse a estos puntos y marque sus coordenadas (que se muestran en la parte superior izquierda) en orden a la distancia al sitio de lanzamiento.


Now zoom in to a launch site and explore its proximity to see if you can easily find any railway, highway, coastline, etc. Move your mouse to these points and mark down their coordinates (shown on the top-left) in order to the distance to the launch site.<br>


In [21]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    # approximate radius of earth in km
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

*TODO:* Mark down a point on the closest coastline using MousePosition and calculate the distance between the coastline point and the launch site.<br>
*TODO:* Marque un punto en la costa más cercana usando MousePosition y calcule la distancia entre el punto de la costa y el lugar de lanzamiento.


In [22]:
# find coordinate of the closet coastline
# e.g.,: Lat: 28.56367  Lon: -80.57163 28.573255 	-80.646895
launch_site_lat = 28.573255
launch_site_lon = -80.64895
coastline_lat = 28.56367
coastline_lon = -80.57163
distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)
distance_coastline

7.628045727510658

In [24]:
# Create and add a folium.Marker on your selected closest coastline point on the map
# Display the distance between coastline point and launch site using the icon property 
# Cree y agregue un folium.Marker en el punto de costa más cercano seleccionado en el mapa
# Muestra la distancia entre el punto de la costa y el sitio de lanzamiento usando la propiedad del icono
# for example
coordinate=[28.57468,-80.65229]
distance_marker = folium.Marker(
    coordinate,
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(distance_coastline),
        )
    )

*TODO:* Draw a `PolyLine` between a launch site to the selected coastline point
<br>*TODO:* Dibuje una `PolyLine` entre un sitio de lanzamiento y el punto de la costa seleccionado

In [29]:
# Create a `folium.PolyLine` object using the coastline coordinates and launch site coordinate
# Crea un objeto `folium.PolyLine` usando las coordenadas de la costa y las coordenadas del sitio de lanzamiento
coordinates=[[28.57468,-80.65229],[28.573255 ,-80.646895]]
lines=folium.PolyLine(locations=coordinates, weight=1)
site_map.add_child(lines)

Your updated map with distance line should look like the following screenshot:
<br>Su mapa actualizado con línea de distancia debería verse como la siguiente captura de pantalla:

<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_distance.png">
</center>


*TODO:* Similarly, you can draw a line betwee a launch site to its closest city, railway, highway, etc. You need to use `MousePosition` to find the their coordinates on the map first<br>
*TODO:* De manera similar, puedes dibujar una línea entre un sitio de lanzamiento y su ciudad, ferrocarril, carretera, etc. más cercana. Primero debes usar `MousePosition` para encontrar sus coordenadas en el mapa.


A railway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/railway.png">
</center>


A highway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/highway.png">
</center>


A city map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/city.png">
</center>


In [30]:
# Create a marker with distance to a closest city, railway, highway, etc.
# Draw a line between the marker to the launch site
# Cree un marcador con la distancia a la ciudad, ferrocarril, carretera, etc. más cercana.
# Dibuja una línea entre el marcador y el sitio de lanzamiento.

In [31]:
coordinates=[[28.57367, -80.58472],[28.5248,-80.64],[28.563197, -80.56772],[28.56,-81.38535]]
coordinate=[28.562302,-80.577356]
for x in coordinates:
    lines=folium.PolyLine(locations=[x,coordinate], weight=1)
    site_map.add_child(lines)

    distance_marker = folium.Marker(
        x,
        icon=DivIcon(
            icon_size=(20,20),
            icon_anchor=(0,0),
            html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(calculate_distance(x[0],x[1],coordinate[0] ,coordinate[1])),
        )
    )
    site_map.add_child(distance_marker)
site_map

In [32]:

coordinates=[[28.57367, -80.58472],[28.5248,-80.64],[28.563197, -80.56772],[28.56,-81.38535]]
coordinate=[28.562302,-80.577356]
for x in coordinates:
    lines=folium.PolyLine(locations=[x,coordinate], weight=1)
    site_map.add_child(lines)

    distance_marker = folium.Marker(
        x,
        icon=DivIcon(
            icon_size=(20,20),
            icon_anchor=(0,0),
            html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(calculate_distance(x[0],x[1],coordinate[0] ,coordinate[1])),
        )
    )
    site_map.add_child(distance_marker)
site_map

After you plot distance lines to the proximities, you can answer the following questions easily:

*   Are launch sites in close proximity to railways?
*   Are launch sites in close proximity to highways?
*   Are launch sites in close proximity to coastline?
*   Do launch sites keep certain distance away from cities?

Also please try to explain your findings.

Después de trazar líneas de distancia hasta las proximidades, podrá responder fácilmente a las siguientes preguntas:

¿Están los sitios de lanzamiento muy cerca de los ferrocarriles?
¿Están los sitios de lanzamiento muy cerca de las autopistas?
¿Están los sitios de lanzamiento muy cerca de la costa?
¿Los sitios de lanzamiento mantienen cierta distancia de las ciudades?

También intente explicar sus hallazgos.


# Next Steps:

Now you have discovered many interesting insights related to the launch sites' location using folium, in a very interactive way. Next, you will need to build a dashboard using Ploty Dash on detailed launch records.<br>
Ahora ha descubierto muchas ideas interesantes relacionadas con la ubicación de los sitios de lanzamiento utilizando folium, de una manera muy interactiva. A continuación, deberá crear un panel utilizando Ploty Dash con registros de lanzamiento detallados.


## Authors


[Pratiksha Verma](https://www.linkedin.com/in/pratiksha-verma-6487561b1/)


<!--## Change Log--!>


<!--| Date (YYYY-MM-DD) | Version | Changed By      | Change Description      |
| ----------------- | ------- | -------------   | ----------------------- |
| 2022-11-09        | 1.0     | Pratiksha Verma | Converted initial version to Jupyterlite|--!>


### <h3 align="center"> IBM Corporation 2022. All rights reserved. <h3/>
